# Color transfer: MW2 vs averaged LSOT-Mix / LSOT-SMix

This notebook follows [Delon--Desolneux's color-transfer example](https://github.com/judelo/gmmot/blob/master/python/GMM_OT_color_transfer.ipynb): Renoir → Gauguin, RGB in [0,1], full-covariance GMMs with K0=K1=10, kmeans initialization, n_init=1, and the posterior-weighted Tmean map.

Only the component coupling changes. `gmmot.GW2` is the preserved MW2 baseline. LSOT averages proportionally lifted 1D plans, then evaluates true Gaussian W2-squared costs. This is not the original projected MixSW/SMixW distance.

On Colab, optionally select **Runtime → Change runtime type → GPU**. LSOT and the shared map support CUDA; scikit-learn EM and the original MW2 LP remain on CPU. The notebook reports the actual device. No CUDA speedup is assumed.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Works in Colab, at the repository root, or inside its notebooks/ directory.
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/LSOT_GMM'), Path.cwd() / 'LSOT_GMM']
repo = next((p for p in candidates if (p / 'pyproject.toml').exists() and (p / 'gmmot.py').exists()), None)
if repo is None:
    repo = Path('/content/LSOT_GMM') if Path('/content').exists() else Path.cwd() / 'LSOT_GMM'
    subprocess.run(['git', 'clone', 'https://github.com/tmp0810/LSOT_GMM.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
print('Repository:', repo)


## Choose the setting

`SMOKE=True` performs a quick correctness run on resized images and a subset of fitting pixels. Set it to **False** for the original image resolution and all pixels in EM. The full configuration uses L=10,50,100. Each smaller budget uses a prefix of the same bank, and Mix/SMix share mean and mixing directions. Seeds below control EM; projection and evaluation seeds are stored separately.


In [ ]:
import json
from experiments.color_transfer.run import Config, run_experiment

SMOKE = True
filename = 'smoke.json' if SMOKE else 'mw2_reference.json'
settings = json.loads((repo / 'experiments/color_transfer/configs' / filename).read_text())
settings['device'] = 'auto'  # 'cpu' for a CPU comparison; 'cuda:0' to require CUDA
# settings['seeds'] = [0, 1, 2]
# settings['projection_counts'] = [10, 50, 100]
# settings['source'] = '/content/my_source.png'
# settings['target'] = '/content/my_target.png'
config = Config(**settings)
print(json.dumps(settings, indent=2))


In [ ]:
rows = run_experiment(config)


## Compare outputs

The first comparison contains unfiltered Tmean outputs (clipped for display). The second applies the same optional guided filter to every method: source + guided_filter(Tmean(source) - source, source), radius 10, epsilon 1e-4. Filtering uses the un-clipped displacement. The map is a barycentric assignment and need not push the fitted source GMM exactly onto the target.


In [ ]:
from IPython.display import display, Image
output = Path(config.output_dir)
seed_dir = output / f'seed_{config.seeds[0]}'
display(Image(filename=str(seed_dir / 'comparison.png')))
if config.guided_filter:
    display(Image(filename=str(seed_dir / 'comparison_guided.png')))


## Numeric results

Transport runtime includes projection evaluation/sorting/lifting plus true Gaussian cost evaluation for LSOT; it includes dense Gaussian costs and the original CPU LP for MW2. Random direction banks are prepared before timing and their cost is reported separately. The map setup/application code is shared. Downloads, plotting, file I/O and evaluation are excluded. CUDA wall times are synchronized.

`color_sw2` measures the output color distribution against the target using a separate shared bank of RGB directions. Lower is better. `relative_cost_gap` compares the unregularized component cost with MW2. Plan RMSE refers to one MW2 LP solution, which need not be unique. These are different notions of quality.

`metrics.csv` has individual seeds and timing repetitions' means/stds; `paper_results.tsv` summarizes across data seeds. Empty across-seed std means only one seed was run.


In [ ]:
import csv
from html import escape
from IPython.display import HTML
columns = ['method', 'L', 'transport_ms_mean', 'map_setup_ms_mean', 'map_apply_ms_mean',
           'relative_cost_gap', 'plan_rmse', 'color_sw2', 'guided_color_sw2']
with (output / 'metrics.csv').open() as stream:
    data = list(csv.DictReader(stream))
html = '<table><tr>' + ''.join(f'<th>{escape(c)}</th>' for c in columns) + '</tr>'
for row in data:
    html += '<tr>' + ''.join(f'<td>{escape(row[c])}</td>' for c in columns) + '</tr>'
display(HTML(html + '</table>'))
print('All artifacts:', output.resolve())


## Inspect a saved plan

The sparse files contain rows, cols, mass, and shape. Densification below is only for inspecting this small K=10 example, not required by the LSOT solver. GMM parameters, the complete projection bank, and evaluation indices/directions are saved beside the plan for reproducibility.


In [ ]:
import numpy as np
saved = np.load(seed_dir / f'LSOT-Mix_L{max(config.projection_counts)}_plan.npz')
P = np.zeros(tuple(saved['shape']))
np.add.at(P, (saved['rows'], saved['cols']), saved['mass'])
gmms = np.load(seed_dir / 'gmms.npz')
np.testing.assert_allclose(P.sum(axis=1), gmms['alpha'], atol=1e-8)
np.testing.assert_allclose(P.sum(axis=0), gmms['beta'], atol=1e-8)
print('Plan shape:', P.shape, 'total mass:', P.sum())
print(P)
